In [1]:
import pickle
import pandas as pd

In [2]:
model_path  = '../2_model_development/model_development_results_dict.pkl'
data_prefix = '../0_data/processed_data/'
non_feature_cols = ['SMILES', 'MP', 'Type', 'Ro5']

In [3]:

with open(model_path, 'rb') as f:
    model_development_results_dict = pickle.load(f)

model_development_results_dict

{'RF': {'trial_results': {0: {'fold_rmses': [78.89326444190277,
     55.221535470548076,
     54.23475484733014,
     67.06245095767079,
     39.1878404173412,
     29.85645210042546,
     28.544213676330273,
     68.68157715446698,
     46.77437650867193,
     32.84614641856789],
    'mean_rmse': 50.13026119932555,
    'std_rmse': 16.756051481327436},
   1: {'fold_rmses': [86.59103627408943,
     51.302040855151176,
     50.81242478738019,
     63.97408713455525,
     39.72535876322593,
     30.72551753791055,
     38.35434830979709,
     74.24010961643346,
     49.75829394249595,
     39.88191856868744],
    'mean_rmse': 52.536513578972645,
    'std_rmse': 16.65805336111435},
   2: {'fold_rmses': [85.50098917411353,
     52.52817693241041,
     51.99230023553815,
     65.16163838904991,
     39.39119570736494,
     28.860797898949638,
     36.307594600234175,
     71.27321656300298,
     48.76417475570707,
     36.51799139984392],
    'mean_rmse': 51.62980756562147,
    'std_rmse': 1

In [ ]:
model_name = 'RF'


In [17]:
model = model_development_results_dict[model_name]['best_model_info']['model']

In [18]:
data = pd.read_parquet(data_prefix + f"data_with_selected_features_{model_name}_scaled.parquet")

In [19]:
data_test = data[data['Type']=='Test']
data_test

,SMILES,MP,Type,Ro5,RDKit_NumHDonors,RDKit_RingCount,RDKit_TPSA,RDKit_fr_Ar_COO,MACCS_165,RDKit_NHOHCount,...,RDKit_SMR_VSA3,RDKit_FpDensityMorgan3,RDKit_BCUT2D_MWHI,RDKit_SMR_VSA4,RDKit_BCUT2D_MRHI,RDKit_NOCount,RDKit_SMR_VSA5,RDKit_PEOE_VSA5,RDKit_MinPartialCharge,MACCS_163
12054,[O-][N+](=O)c1c(C)c(C(=O)C)c(c(c1C(C)(C)C)[N+]...,135.5,Test,1,-0.857454,-0.677491,1.469587,-0.218675,0.307772,-0.839721,...,-0.584558,-1.430020,-0.523016,-0.422994,-0.352389,1.443016,0.998128,-0.289939,0.793927,0.395296
12055,CN(NC(=O)CCC(=O)O)C,154.5,Test,1,0.949337,-1.446897,0.507304,-0.218675,-3.249161,0.694148,...,1.505282,0.089508,-0.535447,-0.422994,-0.486108,0.620346,-0.226208,-0.289939,-0.820140,-2.529749
12056,CCCCc1ccc(cc1)NC(=O)Oc1ccc(cc1)OC,143.0,Test,1,0.045942,0.091915,-0.122990,-0.218675,0.307772,-0.072787,...,-0.584558,0.002796,-0.526874,-0.422994,-0.444646,0.209012,0.252697,-0.289939,-0.954495,0.395296
12057,OC(=O)COCCN1C(=O)c2c(C1=O)cccc2,128.0,Test,1,0.045942,0.091915,0.914654,-0.218675,0.307772,-0.072787,...,0.396785,-0.006839,-0.529202,-0.422994,-0.208970,1.031681,-0.687065,-0.289939,-0.805567,0.395296
12058,CCCCCCCCCCCCCCC,10.0,Test,1,-0.857454,-1.446897,-1.480634,-0.218675,-3.249161,-0.839721,...,-0.584558,-2.847146,-0.645797,-0.422994,-1.374480,-1.436327,2.805458,-0.289939,2.774133,-2.529749
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17215,c1ccc2c(c1)c1cc3ccc4c(c3nc1cc2)cccc4,226.0,Test,1,-0.857454,2.400134,-1.112677,-0.218675,0.307772,-0.839721,...,0.413622,-0.430765,-0.613496,-0.422994,-0.259144,-1.024993,-0.687065,-0.289939,1.203318,0.395296
17216,COc1cc(OC)cc(c1C#N)OC,142.0,Test,1,-0.857454,-0.677491,-0.011090,-0.218675,0.307772,-0.839721,...,-0.584558,-0.839552,-0.529841,-0.422994,-0.642258,0.209012,-0.687065,-0.289939,-0.952036,0.395296
17217,OCCCCCCCCCCc1ccccc1,36.0,Test,1,0.045942,-0.677491,-0.903150,-0.218675,0.307772,-0.072787,...,-0.584558,-0.879629,-0.540789,-0.422994,-0.927824,-1.024993,1.386791,-0.289939,-0.086802,0.395296
17218,Clc1c2OC3Cc4c(C3Oc2c(c(c1Cl)Cl)Cl)cccc4,159.0,Test,1,-0.857454,1.630727,-0.953676,-0.218675,0.307772,-0.839721,...,-0.584558,-0.158241,0.350793,-0.422994,0.004288,-0.613658,-0.018522,-0.289939,-0.811593,0.395296


In [20]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score


def model_evaluation(data_test, model):
    """
    Generate predictions and plot model performance.

    Parameters
    ----------
    data_test : pd.DataFrame  – must contain 'SMILES', 'MP', 'Type', 'Ro5'
    model     : fitted sklearn-compatible regressor

    Returns
    -------
    result : pd.DataFrame – copy of data_test with 'MP_pred' column added
    """
    non_feature_cols = ['SMILES', 'MP', 'Type', 'Ro5']

    result = data_test.copy()
    X = result.drop(columns=non_feature_cols)
    result['MP_pred'] = model.predict(X)

    # ── Compute metrics for each subset ───────────────────────────────
    subsets = {
        'Ro5 = 1':  result[result['Ro5'] == 1],
        'Ro5 = 0':  result[result['Ro5'] == 0],
        'Overall':  result,
    }

    labels, rmses, r2s = [], [], []
    for label, subset in subsets.items():
        y_true = subset['MP'].values
        y_pred = subset['MP_pred'].values
        rmses.append(np.sqrt(mean_squared_error(y_true, y_pred)))
        r2s.append(r2_score(y_true, y_pred))
        labels.append(label)

    colors = ['#4C72B0', '#DD8452', '#55A868']

    # ── Figure 1: RMSE ────────────────────────────────────────────────
    fig1, ax1 = plt.subplots(figsize=(6, 4))
    bars1 = ax1.bar(labels, rmses, color=colors, edgecolor='white', width=0.5)
    for bar, val in zip(bars1, rmses):
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=11)
    ax1.set_ylabel('RMSE (°C)', fontsize=12)
    ax1.set_title('Model RMSE by Ro5 Group', fontsize=13)
    ax1.set_ylim(0, max(rmses) * 1.2)
    ax1.grid(axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

    # ── Figure 2: R² ──────────────────────────────────────────────────
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    bars2 = ax2.bar(labels, r2s, color=colors, edgecolor='white', width=0.5)
    for bar, val in zip(bars2, r2s):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=11)
    ax2.set_ylabel('R²', fontsize=12)
    ax2.set_title('Model R² by Ro5 Group', fontsize=13)
    ax2.set_ylim(0, 1.1)
    ax2.axhline(y=1, color='grey', linestyle='--', linewidth=0.8)
    ax2.grid(axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

    return result

In [21]:
model_evaluation(data_test, model)


ValueError: feature_names mismatch: ['Ro5', 'RDKit_NumHDonors', 'RDKit_RingCount', 'RDKit_TPSA', 'RDKit_fr_Ar_COO', 'MACCS_165', 'RDKit_NHOHCount', 'RDKit_BertzCT', 'RDKit_fr_NH1', 'MACCS_155', 'MACCS_101', 'MACCS_126', 'MACCS_43', 'RDKit_NumHeteroatoms', 'RDKit_NumRotatableBonds', 'RDKit_fr_unbrch_alkane', 'MACCS_131', 'RDKit_NumAmideBonds', 'RDKit_FractionCSP3', 'MACCS_98', 'RDKit_SlogP_VSA8', 'RDKit_NumAliphaticRings', 'RDKit_NumHeterocycles', 'RDKit_fr_COO', 'MACCS_142', 'RDKit_fr_bicyclic', 'MACCS_92', 'RDKit_NumBridgeheadAtoms', 'RDKit_fr_nitrile', 'RDKit_SMR_VSA10', 'RDKit_NumAtomStereoCenters', 'RDKit_fr_imidazole', 'RDKit_fr_Ar_NH', 'RDKit_fr_para_hydroxylation', 'RDKit_NumAromaticCarbocycles', 'RDKit_SlogP_VSA1', 'RDKit_fr_allylic_oxid', 'RDKit_fr_furan', 'RDKit_MinAbsPartialCharge', 'RDKit_SlogP_VSA2', 'RDKit_VSA_EState1', 'RDKit_MaxPartialCharge', 'RDKit_VSA_EState2', 'RDKit_PEOE_VSA7', 'RDKit_SMR_VSA3', 'RDKit_FpDensityMorgan3', 'RDKit_BCUT2D_MWHI', 'RDKit_SMR_VSA4', 'RDKi